In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

# ==========================================
# Getting Started: Reloading and Transformations
# ==========================================
print("--- Loading Data and Applying Transformations ---")
if not os.path.exists("train.csv"):
    raise FileNotFoundError("🛑 STOP! 'train.csv' is missing.")

df = pd.read_csv("train.csv", parse_dates=["datetime"])
serie = df.set_index("datetime")["count"].sort_index().asfreq("h").ffill()

# Apply Log and Differencing (from TP6.2)
serie_log = np.log(serie + 1)
serie_diff24_1 = serie_log.diff(24).diff(1).dropna()

# ==========================================
# Step 1: Train/Test Split
# ==========================================
print("--- Step 1: Train/Test Split (Last 7 Days) ---")
test_size = 168 # 7 days * 24 hours

# Split the raw series
train_serie, test_serie = serie.iloc[:-test_size], serie.iloc[-test_size:]
# Split the log series
train_log, test_log = serie_log.iloc[:-test_size], serie_log.iloc[-test_size:]
# Split the differenced series
train_diff, test_diff = serie_diff24_1.loc[:train_serie.index[-1]], serie_diff24_1.loc[test_serie.index[0]:]

# Helper function to print metrics
def print_metrics(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"Metrics for {model_name}:")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE:  {mae:.2f}\n")
    return rmse, mae

results = {}

# ==========================================
# Step 2: Naive Baseline Model
# ==========================================
print("--- Step 2: Seasonal Naive Baseline ---")
# The best naive guess for hourly data is what happened exactly 24 hours ago
pred_naive = serie.shift(24).loc[test_serie.index]

results["Seasonal Naive"] = print_metrics(test_serie, pred_naive, "Seasonal Naive (24h shift)")

# ==========================================
# Step 3: AR(p) Model on Stationary Data
# ==========================================
print("--- Step 3: AR Model with Re-integration ---")
# Train AR on the differenced data (using lag 1, 2, and 3)
ar_model = AutoReg(train_diff, lags=3).fit()

# Predict on the test set
pred_diff_ar = ar_model.predict(start=test_diff.index[0], end=test_diff.index[-1], dynamic=False)

# Re-integrate mathematically (Reverse diff(1) and diff(24))
log_t1 = serie_log.shift(1).loc[test_serie.index]
log_t24 = serie_log.shift(24).loc[test_serie.index]
log_t25 = serie_log.shift(25).loc[test_serie.index]

pred_log_ar = pred_diff_ar + log_t1 + log_t24 - log_t25
# Reverse the Log (exp - 1)
pred_ar = np.exp(pred_log_ar) - 1

results["AR(3) Transformed"] = print_metrics(test_serie, pred_ar, "AR(3) Model")

# ==========================================
# Step 4: SARIMA Model
# ==========================================
print("--- Step 4: SARIMA Model ---")
# To make this run fast, we only train SARIMA on the last 4 weeks of training data
train_log_subset = train_log.iloc[- (24 * 28):]

# ARIMA handles the first difference (d=1), and Seasonal handles the 24h cycle (D=1, s=24)
sarima_model = ARIMA(train_log_subset, order=(1, 1, 1), seasonal_order=(0, 1, 1, 24))
sarima_fit = sarima_model.fit()

pred_log_sarima = sarima_fit.predict(start=test_log.index[0], end=test_log.index[-1], dynamic=False)
pred_sarima = np.exp(pred_log_sarima) - 1

results["SARIMA(1,1,1)x(0,1,1,24)"] = print_metrics(test_serie, pred_sarima, "SARIMA Model")

# ==========================================
# Step 5: Visual Comparison
# ==========================================
print("--- Step 5: Visualizing Predictions ---")
plt.figure(figsize=(15, 6))
plt.plot(test_serie.index, test_serie, label='Actual Demand', color='black', linewidth=2)
plt.plot(test_serie.index, pred_naive, label='Seasonal Naive', color='gray', linestyle='--')
plt.plot(test_serie.index, pred_ar, label='AR(3) Re-integrated', color='blue', alpha=0.7)
plt.plot(test_serie.index, pred_sarima, label='SARIMA', color='red', alpha=0.7)

plt.title("Rolling Forecast Comparison (Last 7 Days)")
plt.xlabel("Date")
plt.ylabel("Hourly Bike Rentals")
plt.legend()
plt.tight_layout()
plt.show()

# Final Metrics Table
print("--- Final Metrics Summary ---")
df_metrics = pd.DataFrame(results, index=["RMSE", "MAE"]).T
display(df_metrics.sort_values(by="RMSE"))